# Data Cleaning & Reporting Automation
**Project:** Automated Sales Data Cleaning, Analysis and Reporting


## 1. Objective
Clean a messy sales dataset, automate preprocessing, validate data quality, calculate KPIs, and generate visual summaries for reporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('../data/raw_sales_data.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
display(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 2. Data Cleaning
Steps: convert data types, handle missing values, standardize text, correct invalid numeric values, remove duplicates, and validate the result.

In [ ]:
def clean_data(df):
    df = df.copy()
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce', dayfirst=True)
    df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
    df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
    df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce')

    df['Customer_Name'] = df['Customer_Name'].fillna('Unknown Customer')
    df['Region'] = df['Region'].fillna('Unknown').astype(str).str.strip().str.title()

    valid_qty = df.loc[df['Quantity'] > 0, 'Quantity']
    df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
    df['Quantity'] = df['Quantity'].fillna(valid_qty.median()).round().astype(int)

    df['Sales'] = df['Sales'].mask(df['Sales'] < 0)
    df['Sales'] = df['Sales'].fillna(df.groupby('Category')['Sales'].transform('median'))
    df['Sales'] = df['Sales'].fillna(df['Sales'].median())

    df['Profit'] = df['Profit'].mask(df['Profit'] < 0)
    df['Profit'] = df['Profit'].fillna(df.groupby('Category')['Profit'].transform('median'))
    df['Profit'] = df['Profit'].fillna(df['Profit'].median())

    df = df.drop_duplicates()
    df['Order_Date'] = df['Order_Date'].fillna(df['Order_Date'].median())
    df['Sales'] = df['Sales'].round(2)
    df['Profit'] = df['Profit'].round(2)
    return df.sort_values('Order_Date').reset_index(drop=True)

cleaned_df = clean_data(df)
cleaned_df.head()

In [ ]:
print('Cleaned shape:', cleaned_df.shape)
print('Missing values after cleaning:', cleaned_df.isnull().sum().sum())
print('Duplicate rows after cleaning:', cleaned_df.duplicated().sum())
print('Invalid quantities:', (cleaned_df['Quantity'] <= 0).sum())
print('Negative sales:', (cleaned_df['Sales'] < 0).sum())

In [ ]:
cleaned_df.to_csv('../data/cleaned_sales_data.csv', index=False)
print('Cleaned dataset exported successfully.')

## 3. KPI Analysis

In [ ]:
total_sales = cleaned_df['Sales'].sum()
total_profit = cleaned_df['Profit'].sum()
total_orders = cleaned_df['Order_ID'].nunique()
total_quantity = cleaned_df['Quantity'].sum()
aov = total_sales / total_orders

print(f'Total Sales: ₹{total_sales:,.2f}')
print(f'Total Profit: ₹{total_profit:,.2f}')
print(f'Total Orders: {total_orders:,}')
print(f'Total Quantity: {total_quantity:,}')
print(f'Average Order Value: ₹{aov:,.2f}')

## 4. Visual Reporting

In [ ]:
cleaned_df['Month'] = cleaned_df['Order_Date'].dt.to_period('M').astype(str)
monthly = cleaned_df.groupby('Month')['Sales'].sum()
plt.figure(figsize=(10,5))
plt.plot(monthly.index, monthly.values, marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Sales Trend')
plt.xlabel('Month'); plt.ylabel('Sales')
plt.tight_layout(); plt.show()

In [ ]:
category = cleaned_df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,5))
category.plot(kind='bar')
plt.title('Sales by Category')
plt.xlabel('Category'); plt.ylabel('Sales')
plt.tight_layout(); plt.show()

In [ ]:
region = cleaned_df.groupby('Region')['Profit'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,5))
region.plot(kind='bar')
plt.title('Profit by Region')
plt.xlabel('Region'); plt.ylabel('Profit')
plt.tight_layout(); plt.show()

In [ ]:
top10 = cleaned_df.groupby('Product')['Sales'].sum().sort_values(ascending=False).head(10).sort_values()
plt.figure(figsize=(9,5))
top10.plot(kind='barh')
plt.title('Top 10 Products by Sales')
plt.xlabel('Sales')
plt.tight_layout(); plt.show()

## 5. Conclusion
The workflow transforms a messy sales dataset into a validated dataset and produces automated KPIs and visual summaries. The cleaned CSV can be imported into Power BI to build an interactive dashboard.